<a href="https://colab.research.google.com/github/filipchudzynski/stock-market-non-gaussianity-analyzer_v2/blob/main/log_energy_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! git clone https://github.com/filipchudzynski/stock-market-non-gaussianity-analyzer_v2.git

Cloning into 'stock-market-non-gaussianity-analyzer_v2'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 150 (delta 61), reused 73 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 24.51 MiB | 6.34 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [1]:
from IPython.display import Javascript
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})'''))


<IPython.core.display.Javascript object>

In [2]:
import sys
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/log_energy")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/models")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/tests")

from models.white_noise import white_noise
from models.brownian_motion import brownian_motion
from log_energy.operators import increment_operator
from log_energy.energy import log_energy_field, sliding_baseline,local_energy
from log_energy.mi_knn import mi_knn
from log_energy.intermittency import intermittency_variance
from log_energy.covariance import log_energy_covariance

In [7]:
N=5000
s=64
kappa=10
max_lag=200

def log_field_energy_computations(signal, s, kappa):
  window = max(int(kappa * s), 5)
  E = local_energy(signal)
  baseline = sliding_baseline(E, window)
  log_field = log_energy_field(signal,s,kappa)
  log_field_var = intermittency_variance(log_field)
  return E, baseline, log_field, log_field_var

wn = white_noise(N)
bm = brownian_motion(N)
def validate_log_energy(signal,s,kappa,max_lag):
  E, baseline,log_field,log_field_var = log_field_energy_computations(signal, s, kappa)
  cov = log_energy_covariance(log_field,max_lag)


  increments = increment_operator(signal, s)
  E_inc, baseline_inc, log_field_increments, log_field_var_inc = log_field_energy_computations(increments, s,kappa)
  cov_inc = log_energy_covariance(log_field_increments,max_lag)


  mkr_size = 3
  opacity = 0.7
  marker=dict(line=dict(width=2))
  fig1 = make_subplots(rows=1, cols=3,
                         subplot_titles=["signal, energy, baseline","increments, energy, baseline", f"log-energy field [var: {log_field_var:2.3f}, {log_field_var_inc:2.3f}(inc)]"])
  fig1.add_trace(go.Scatter(y=signal,opacity=opacity, mode="lines",  marker=marker, name="signal"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=E,opacity=opacity, mode="lines",   marker=marker,name="E"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=baseline, mode="lines", name="baseline"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=increments,opacity=opacity, mode="lines",   marker=marker,name="increments"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=E_inc,opacity=opacity, mode="lines",  marker=marker, name="E_inc"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=baseline_inc, mode="lines", name="baseline_inc"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=log_field,opacity=opacity, mode="lines",  marker=marker, name="log energy field"), row=1, col=3)
  fig1.add_trace(go.Scatter(y=log_field_increments,opacity=opacity, mode="lines",  marker=marker, name="log energy field(inc)"), row=1, col=3)

  fig1.update_layout(height=600, title="comparison of local energy, baseline and log energy field computed from raw signal and increments")
  fig1.show()

  fig2 = go.Figure()
  fig2.update_layout(title="Comparison of covariance from log energy field calculated using increments and raw signal")
  fig2.add_trace(go.Scatter(y=cov,mode="lines",name="Covariance"))
  fig2.add_trace(go.Scatter(y=cov_inc,mode="lines",name="Covariance increments"))
  fig2.show()



# Explanation
In those plots below I validate the internals of the library and assess it's impact on intermittency.
On the first two plots, first row I display behavior of local energy and baseline for raw signal and increments. On the 3rd one I display comparison of log energy field for both increments and raw signal, additionally variance for log-energy is displayed in the title.

In the second row comparison of covariance is displayed.

#white noise
For these plots nothing unexpected occurs, variance correspond to average magnitutde of fluctuations(~-5) and covariance is close to 0 for all the lags,as expected.



In [8]:
validate_log_energy(wn,s=s,kappa=kappa,max_lag=max_lag)

#Brownian Motion
In the first plot(first row) for brownian motion I observe, that baseline serves as detrending mechanism in the log_energy_field function. It follows the trend of the energy relatively well, however it depends on how small is the s parameter. On the 2nd plot, displaying baseline and energy for increments it's less obvious, whether baseline is not introducing some artifacts.

On the 3rd plot I note unexpected artifact in the beginning of the log energy field(raw signal). I expect it's related to the fact, that brownian motion starts in 0 and removal of the baseline introduces this large fluctuation in log energy field.

Finally, I note that the variance for raw signal is not comparable with white noise(5.074 vs 0.927), on the contrary variance of increments is comparable(5.216 vs 4.505). I can conclude, that increments could be used to compare variance.

In the 2nd row we see unexpectedly, that for both signal the correlation is present. The covariance is not comparable with white noise for both increments and raw signal, especially for increments, we see a large spike for short lags.



In [11]:
validate_log_energy(bm,s=s,kappa=kappa,max_lag=max_lag)